# News Recommendation — Popularity Debiased MIECL

This notebook is configured to run code from the cloned GitHub repository `MIECL-aug-popularityBiasElimination` while loading the massive dataset from Google Drive to avoid re-downloading it every session.

## Required Structure
**In Colab Local Storage (`/content/`):**
The repo is cloned here. The notebook expects it at `/content/MIECL-aug-popularityBiasElimination`.

**On Google Drive (`/content/drive/MyDrive/...`):**
You must have the dataset on your Drive. By default, it expects:
```
My Drive/
└── News Recc Code/
    └── dataset/
        ├── MINDsmall_train/
        ├── MINDsmall_dev/
        └── glove/
```


## Cell 1 — Setup Paths and Mount Drive

In [ ]:
from google.colab import drive
import os
import sys

drive.mount('/content/drive')

# ── CONFIGURE THESE PATHS ─────────────────────────────────────────────────────
# Where the code is (cloned from Git)
CODE_DIR = '/content/MIECL-aug-popularityBiasElimination'

# Where the dataset is stored on your Google Drive
DATA_DIR = '/content/drive/MyDrive/News Recc Code/dataset'
# ──────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)

print('Working directory:', os.getcwd())
print('Files in code dir:', os.listdir(CODE_DIR))

## Cell 2 — Install dependencies

In [ ]:
# NLTK punkt tokenizer (used in data_process.py)
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

# Verify PyTorch sees the GPU
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Cell 3 — Create dummy.txt (required by DataProcess)

In [ ]:
# main.py passes file6='dummy.txt' to DataProcess.
dummy_path = os.path.join(CODE_DIR, 'dummy.txt')
if not os.path.exists(dummy_path):
    open(dummy_path, 'w').close()
    print('Created dummy.txt in code dir.')
else:
    print('dummy.txt already exists.')

## Cell 4 — Verify dataset paths on Drive

In [ ]:
paths_to_check = [
    os.path.join(DATA_DIR, 'MINDsmall_train', 'news.tsv'),
    os.path.join(DATA_DIR, 'MINDsmall_train', 'behaviors.tsv'),
    os.path.join(DATA_DIR, 'MINDsmall_train', 'entity_embedding.vec'),
    os.path.join(DATA_DIR, 'MINDsmall_dev',   'news.tsv'),
    os.path.join(DATA_DIR, 'MINDsmall_dev',   'behaviors.tsv'),
    os.path.join(DATA_DIR, 'MINDsmall_dev',   'entity_embedding.vec'),
    os.path.join(DATA_DIR, 'glove', 'glove.840B.300d.txt'),
]

all_ok = True
for p in paths_to_check:
    exists = os.path.exists(p)
    status = '\u2713' if exists else '\u2717 MISSING'
    print(f'{status}  {p}')
    if not exists:
        all_ok = False

if all_ok:
    print('\nAll dataset files found — ready to run.')
else:
    print('\nFix the missing dataset files on your Drive (or update DATA_DIR in Cell 1).')

## Cell 5 — Patch file paths (Code -> Colab, Data -> Drive)

This automatically edits `data_process.py` and `main.py` in the cloned repo to load the dataset from your Drive and save outputs to the local Colab disk.

In [ ]:
# ── Patch data_process.py ────────────────────────────────────────────────────
dp_path = os.path.join(CODE_DIR, 'data_process.py')
with open(dp_path, 'r', encoding='utf-8') as f:
    src = f.read()

old_train_ent = "open('/content/drive/MyDrive/News Recc Code/dataset/MINDsmall_train/entity_embedding.vec'"
new_train_ent = f"open(r'{DATA_DIR}/MINDsmall_train/entity_embedding.vec'"
old_dev_ent   = "open('/content/drive/MyDrive/News Recc Code/dataset/MINDsmall_dev/entity_embedding.vec'"
new_dev_ent   = f"open(r'{DATA_DIR}/MINDsmall_dev/entity_embedding.vec'"

src = src.replace(old_train_ent, new_train_ent).replace(old_dev_ent, new_dev_ent)
if 'import os' not in src:
    src = 'import os\n' + src

with open(dp_path, 'w', encoding='utf-8') as f:
    f.write(src)
print('data_process.py  — entity paths patched to Drive.')

# ── Patch main.py ────────────────────────────────────────────────────────────
PRESERVE = os.path.join(CODE_DIR, 'outputs')
os.makedirs(PRESERVE, exist_ok=True)

main_path = os.path.join(CODE_DIR, 'main.py')
with open(main_path, 'r', encoding='utf-8') as f:
    main_src = f.read()

replacements = [
    ("file1 = 'MINDsmall_train/news.tsv'",
     f"file1 = r'{DATA_DIR}/MINDsmall_train/news.tsv'"),
    ("file2 = 'MINDsmall_dev/news.tsv'",
     f"file2 = r'{DATA_DIR}/MINDsmall_dev/news.tsv'"),
    ("file3 = 'MINDsmall_train/behaviors.tsv'",
     f"file3 = r'{DATA_DIR}/MINDsmall_train/behaviors.tsv'"),
    ("file4 = 'MINDsmall_dev/behaviors.tsv'",
     f"file4 = r'{DATA_DIR}/MINDsmall_dev/behaviors.tsv'"),
    ("file5 = 'glove/glove.840B.300d.txt'",
     f"file5 = r'{DATA_DIR}/glove/glove.840B.300d.txt'"),
    ("file6 = 'dummy.txt'",
     f"file6 = r'{CODE_DIR}/dummy.txt'"),
    ("default='C:/Users/anany/Desktop/Ananya/2023/Estonia Projects/News Recc/MIECL-master'",
     f"default=r'{PRESERVE}'"),
]

for old, new in replacements:
    if old in main_src:
        main_src = main_src.replace(old, new)
        print(f'main.py  — patched: {old[:55]}...')

with open(main_path, 'w', encoding='utf-8') as f:
    f.write(main_src)
print(f'\nOutputs will be saved locally in Colab at: {PRESERVE}')
print('NOTE: If your Colab session ends, you will lose the outputs. Download them if needed.')

## Cell 6 — Uncomment the training loop

In [ ]:
main_path = os.path.join(CODE_DIR, 'main.py')
with open(main_path, 'r', encoding='utf-8') as f:
    main_src = f.read()

TRAIN_OPEN  = "        '''\n        for n_ep"
TRAIN_CLOSE = "        del train_candidate, train_user, train_label\n    '''"

if TRAIN_OPEN in main_src:
    main_src = main_src.replace(
        TRAIN_OPEN, "        \n        for n_ep"
    ).replace(
        TRAIN_CLOSE, "        del train_candidate, train_user, train_label\n"
    )
    with open(main_path, 'w', encoding='utf-8') as f:
        f.write(main_src)
    print('Training loop uncommented — ready to train.')
else:
    print('Training loop already active.')

## Cell 7 — Run Training + Validation

In [ ]:
!python "{CODE_DIR}/main.py" \
    --num_epoch 1 \
    --num_dataset 1 \
    --batch_size 30 \
    --hid_dim 400 \
    --num_head 20 \
    --num_prototype 5 \
    --alpha 1.0 \
    --dropout_rate 0.1 \
    --multi_rep_mode concat \
    --infonce_mode prototype_self \
    --contrastive_mode USER \
    --gnn_mode nogat \
    --agg_mode soft

## Cell 8 — View Results

In [ ]:
import glob
PRESERVE = os.path.join(CODE_DIR, 'outputs')
score_files = sorted(glob.glob(os.path.join(PRESERVE, 'scores_*.txt')))

if not score_files:
    print('No score files found. Run Cell 7 first.')
else:
    for sf in score_files:
        print(f'\n=== {os.path.basename(sf)} ===')
        with open(sf) as f:
            print(f.read())